# Explore GeoLibre from Python

`Welcome.ipynb` covers the basics: moving the camera and adding markers and GeoJSON. This notebook goes further with raster and vector data, working through visualization and analysis for each:

- **Part 1, raster:** tile layers, a Cloud Optimized GeoTIFF, terrain analysis with NumPy, and converting raster results to vector.
- **Part 2, vector:** live earthquake data, data-driven styles, heatmaps, labels, a GeoPandas spatial join and 3D extrusions.
- **Part 3, desktop:** managing layers by id and running Whitebox tools, processing tools and a Model Builder graph.

Run the cells from top to bottom with **Shift+Enter**. The map beside this notebook updates as you go. Part 1 comes first because new layers draw on top, which leaves the vector layers above the rasters.

**Web or desktop?** The web build runs Python in your browser and sends commands to the map one way, so it cannot return layer ids. Part 3 needs ids, so it only runs in the desktop app; on the web, those cells print a note and skip their work. Parts 1 and 2 run in both. The first run downloads a few packages and can take a minute or two.

## Setup

`is_connected()` tells you whether a GeoLibre window is listening. `fetch_bytes` downloads a file in either runtime: Pyodide uses the browser's `fetch`, and desktop Python uses `urllib`.

In [ ]:
%pip install rasterio geopandas matplotlib

In [ ]:
import asyncio
import json
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.features import shapes, sieve
from rasterio.io import MemoryFile

import geolibre

m = geolibre.connect()
print("Connected to GeoLibre:", geolibre.is_connected())


async def fetch_bytes(url):
    """Download a file in the browser or desktop kernel."""
    if sys.platform == "emscripten":
        from pyodide.http import pyfetch

        response = await pyfetch(url)
        return await response.bytes()

    from urllib.request import urlopen

    with urlopen(url, timeout=120) as response:
        return response.read()

# Part 1: Raster

## 1.1 Tile layers

`set_basemap` takes any MapLibre style URL. `add_tile_layer` adds XYZ raster tiles as a regular layer, and `bounds` and `minzoom` limit them to one area. This study area is the Sierra Nevada in California.

In [ ]:
SIERRA = [-120.75, 36.64, -117.31, 38.62]

m.set_basemap("https://tiles.openfreemap.org/styles/dark")
m.fit_bounds(SIERRA)

imagery_id = m.add_tile_layer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite imagery",
    attribution="Esri",
    bounds=SIERRA,
    minzoom=5,
)

## 1.2 Cloud Optimized GeoTIFFs

`add_cog` hands a URL to the app, which reads only the parts of the file it needs, so it suits rasters of any size. `bands` picks the red, green and blue bands of this Landsat scene of San Francisco Bay, and the map zooms to it.

In [ ]:
landsat_id = m.add_cog(
    "https://raw.githubusercontent.com/opengeos/data/main/raster/landsat7.tif",
    name="Landsat, San Francisco Bay",
    bands="1,2,3",
)

## 1.3 Read and chart the raster

For analysis, read the data into Python. This is a 90 m SRTM elevation model of the Sierra Nevada. `rasterio` reads from memory, and `out_shape` picks a coarser overview so the arrays stay small. Then it is an ordinary NumPy array, and the notebook can chart it.

In [ ]:
DEM_URL = "https://raw.githubusercontent.com/opengeos/data/main/raster/srtm90.tif"
FACTOR = 4

dem_bytes = await fetch_bytes(DEM_URL)
with MemoryFile(dem_bytes) as memfile, memfile.open() as src:
    shape = (src.height // FACTOR, src.width // FACTOR)
    elevation = src.read(1, out_shape=shape).astype("float32")
    transform = src.transform * src.transform.scale(src.width / shape[1], src.height / shape[0])
    crs = src.crs

print(f"{shape[1]} x {shape[0]} cells, elevation {elevation.min():.0f} to {elevation.max():.0f} m")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.hist(elevation.ravel(), bins=60, color="#0ea5e9")
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Cells")
ax.set_title("Sierra Nevada elevation")
plt.show()

## 1.4 Terrain analysis

Slope and hillshade both come from the elevation gradient. `add_raster` sends a GeoTIFF from memory to the map, so any array you compute can become a layer. It suits results up to a few megabytes. `colormap` and `rescale` control how a single band is drawn, and `nodata` makes cells transparent.

Drawing semi-transparent elevation over a hillshade gives classic shaded relief. On top, this marks only the terrain steeper than 25 degrees.

In [ ]:
def to_geotiff(array, nodata=None):
    """Encode a 2D array as GeoTIFF bytes on the DEM's grid."""
    profile = {
        "driver": "GTiff",
        "height": array.shape[0],
        "width": array.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "compress": "deflate",
        "nodata": nodata,
    }
    with MemoryFile() as memfile:
        with memfile.open(**profile) as dst:
            dst.write(array.astype("float32"), 1)
            # Overviews let the map draw the whole raster when zoomed out.
            dst.build_overviews([2, 4, 8], Resampling.average)
        return memfile.read()


# Cell size in meters. Degrees of longitude shrink toward the poles.
latitude = np.radians(transform.f + transform.e * shape[0] / 2)
cell_y = abs(transform.e) * 111_320
cell_x = transform.a * 111_320 * np.cos(latitude)

dz_dy, dz_dx = np.gradient(elevation, cell_y, cell_x)
slope = np.arctan(np.hypot(dz_dx, dz_dy))

# Light from the northwest, 45 degrees above the horizon.
azimuth, altitude = np.radians(315), np.radians(45)
aspect = np.arctan2(-dz_dx, dz_dy)
hillshade = np.sin(altitude) * np.cos(slope) + np.cos(altitude) * np.sin(slope) * np.cos(
    azimuth - aspect
)
hillshade = np.clip(hillshade, 0, 1) * 255

slope_degrees = np.degrees(slope)
print(f"Mean slope {slope_degrees.mean():.1f} degrees, steepest {slope_degrees.max():.1f} degrees")

steep = np.where(slope_degrees >= 25, slope_degrees, -1)

hillshade_id = m.add_raster(
    to_geotiff(hillshade), name="Hillshade", colormap="gray", rescale=(0, 255)
)
elevation_id = m.add_raster(
    to_geotiff(elevation), name="Elevation", colormap="terrain", rescale=(500, 4000), opacity=0.45
)
steep_id = m.add_raster(
    to_geotiff(steep, nodata=-1),
    name="Slopes over 25 degrees",
    colormap="magma",
    rescale=(25, 45),
    nodata=-1,
)

## 1.5 From raster to vector

Rasters and vectors work well together. Here, each peak's location looks up its elevation in the array (a rough value, since this grid has cells of about 350 m), and `shapes` turns the land above 3,000 m into polygons. `sieve` first removes small patches, which keeps the result simple.

In [ ]:
peaks = {
    "Half Dome": (-119.533, 37.746),
    "Mount Lyell": (-119.271, 37.739),
    "Mammoth Mountain": (-119.032, 37.631),
}

peak_points = []
for name, (lng, lat) in peaks.items():
    row, col = rasterio.transform.rowcol(transform, lng, lat)
    height = float(elevation[row, col])
    peak_points.append({"lng": lng, "lat": lat, "label": f"{name}: {height:.0f} m"})
    print(peak_points[-1]["label"])

high_ground = sieve((elevation >= 3000).astype("uint8"), size=20)
high_polygons = [
    {"type": "Feature", "properties": {}, "geometry": geometry}
    for geometry, value in shapes(high_ground, transform=transform)
    if value == 1
]
print(f"{len(high_polygons)} areas above 3,000 m")

high_id = m.add_geojson(
    {"type": "FeatureCollection", "features": high_polygons},
    name="Above 3,000 m",
    fillColor="#e0f2fe",
    fillOpacity=0.35,
    strokeColor="#f8fafc",
    strokeWidth=1,
)

LABELS = {
    "enabled": True,
    "expression": "",
    "placement": "point",
    "color": "#ffffff",
    "haloColor": "#000000",
    "haloWidth": 1.5,
    "size": 13,
    "anchor": "top",
    "offsetY": 0.8,
}

peaks_id = m.add_geojson(
    {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"label": point["label"]},
                "geometry": {"type": "Point", "coordinates": [point["lng"], point["lat"]]},
            }
            for point in peak_points
        ],
    },
    name="Peaks",
    fillColor="#f59e0b",
    strokeColor="#ffffff",
    strokeWidth=1.5,
    circleRadius=6,
    labels={**LABELS, "field": "label"},
)
m.fly_to(-119.3, 37.7, zoom=9, pitch=45, bearing=20, duration=3000)

# Part 2: Vector

## 2.1 Load remote data and fly away

This is ordinary Python, so you can inspect and trim the data before it reaches the map. The earthquake feed from the [USGS](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php) covers magnitude 4.5 and above for the past 30 days, so the numbers change each time you run it. The country outlines come from [Natural Earth](https://www.naturalearthdata.com/).

`fly_to` animates the camera and can also set the `pitch`, `bearing` and `duration` (in milliseconds).

In [ ]:
QUAKES_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson"
COUNTRIES_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
    "master/geojson/ne_110m_admin_0_countries.geojson"
)

quakes = json.loads(await fetch_bytes(QUAKES_URL))
countries = json.loads(await fetch_bytes(COUNTRIES_URL))

# Keep the fields this notebook uses, so the layers stay small.
for feature in quakes["features"]:
    p = feature["properties"]
    feature["properties"] = {"mag": p["mag"], "place": p["place"], "tsunami": p["tsunami"]}
    # USGS adds depth as a third coordinate. Keep the points on the map's surface.
    feature["geometry"]["coordinates"] = feature["geometry"]["coordinates"][:2]
for feature in countries["features"]:
    feature["properties"] = {"name": feature["properties"]["NAME"]}

magnitudes = [f["properties"]["mag"] for f in quakes["features"]]
print(f"{len(magnitudes)} earthquakes, magnitudes {min(magnitudes)} to {max(magnitudes)}")

m.fly_to(150, 10, zoom=2, pitch=0, bearing=0, duration=3000)

## 2.2 Style layers as you add them

Keyword arguments to `add_geojson` become the layer's style. Here `fillOpacity` and `strokeWidth` give the countries a subtle outline.

In [ ]:
countries_id = m.add_geojson(
    countries,
    name="Countries",
    fillColor="#334155",
    fillOpacity=0.35,
    strokeColor="#94a3b8",
    strokeWidth=0.5,
)

## 2.3 Heatmaps and data-driven styles

`pointRenderer="heatmap"` shows where activity is concentrated. Above it, `vectorStyleMode="graduated"` picks each point's color from `vectorStyleStops`, and `proportionalSizeEnabled` scales its radius with the magnitude.

In [ ]:
heatmap_id = m.add_geojson(
    quakes,
    name="Earthquake density",
    pointRenderer="heatmap",
    heatmapRadius=20,
    heatmapIntensity=0.6,
)

quakes_id = m.add_geojson(
    quakes,
    name="Earthquakes",
    vectorStyleMode="graduated",
    vectorStyleProperty="mag",
    vectorStyleStops=[
        {"value": 4.5, "color": "#fde047", "label": "4.5"},
        {"value": 5.5, "color": "#fb923c", "label": "5.5"},
        {"value": 6.5, "color": "#ef4444", "label": "6.5"},
        {"value": 7.5, "color": "#a21caf", "label": "7.5+"},
    ],
    proportionalSizeEnabled=True,
    proportionalSizeProperty="mag",
    proportionalSizeMinValue=4.5,
    proportionalSizeMaxValue=8,
    proportionalSizeMinRadius=2,
    proportionalSizeMaxRadius=14,
    strokeColor="#0f172a",
    strokeWidth=0.5,
)

## 2.4 Filter and label

Python decides what goes in each layer, so this one holds only the ten strongest earthquakes, each labeled with its magnitude.

In [ ]:
strongest = sorted(quakes["features"], key=lambda f: f["properties"]["mag"], reverse=True)[:10]
for feature in strongest:
    feature["properties"] = {
        **feature["properties"],
        "label": f"M{feature['properties']['mag']:.1f}",
    }

strongest_id = m.add_geojson(
    {"type": "FeatureCollection", "features": strongest},
    name="Ten strongest",
    fillColor="#ffffff",
    strokeColor="#a21caf",
    strokeWidth=2,
    circleRadius=6,
    labels={**LABELS, "field": "label"},
)

## 2.5 Spatial analysis with GeoPandas

Anything with a `__geo_interface__` can go on the map, including a GeoDataFrame. This spatial join counts the earthquakes inside each country and keeps those with at least three. Most earthquakes happen at sea, so only a share of them are counted.

In [ ]:
countries_gdf = gpd.GeoDataFrame.from_features(countries["features"], crs="EPSG:4326")
quakes_gdf = gpd.GeoDataFrame.from_features(quakes["features"], crs="EPSG:4326")

joined = gpd.sjoin(quakes_gdf, countries_gdf, predicate="within")
counts = joined.groupby("name").size().rename("quake_count").reset_index()

summary = countries_gdf.merge(counts, on="name")
summary = summary[summary["quake_count"] >= 3]
summary.sort_values("quake_count", ascending=False)[["name", "quake_count"]].head(10)

`extrusionEnabled` turns the result into a 3D map, with each country's height set by its count.

In [ ]:
summary_id = m.add_geojson(
    summary,
    name="Earthquakes per country",
    fillColor="#f97316",
    strokeColor="#fed7aa",
    extrusionEnabled=True,
    extrusionColor="#f97316",
    extrusionOpacity=0.85,
    extrusionHeightProperty="quake_count",
    extrusionHeightScale=8000,
)
m.fly_to(120, 0, zoom=2.5, pitch=55, bearing=-15, duration=3000)

# Part 3: Desktop

## 3.1 Manage layers

On desktop, each `add_*` call returned the new layer's id, and `list_layers()` reads the map's current state. Pass an id to `set_visibility`, `set_opacity`, `set_style` or `zoom_to_layer` to change a layer after adding it.

In [ ]:
PART3_IDS = (quakes_id, elevation_id, strongest_id, countries_id, heatmap_id, summary_id)
HAS_LAYER_IDS = None not in PART3_IDS
if not HAS_LAYER_IDS:
    print(
        "This kernel did not return layer ids, so Part 3 skips its work. This is normal on the web."
    )
else:
    part3_start = {layer["id"] for layer in m.list_layers()}
    for layer in m.list_layers():
        print(layer["name"], layer["type"], layer["visible"], layer["opacity"])

In [ ]:
if HAS_LAYER_IDS:
    m.set_visibility(summary_id, False)
    m.set_visibility(heatmap_id, False)
    m.set_opacity(countries_id, 0.6)
    m.set_style(strongest_id, circleRadius=9, strokeColor="#22d3ee")
    m.fly_to(pitch=0, bearing=0)
    m.zoom_to_layer(quakes_id)
    print(m.get_layer(heatmap_id))

## 3.2 Run a Whitebox tool on a raster

`run_whitebox_tool` runs a tool from the Whitebox toolbox on a raster layer, right in the app. This one derives slope from the elevation layer, and the `.tif` output tells it to return a raster. The call returns before the tool finishes, so `wait_for_new_layer` polls `list_layers()` until the result appears.

In [ ]:
async def wait_for_new_layer(name, known, timeout=60):
    """Return the id of a layer called ``name`` whose id is not in ``known``."""
    for _ in range(timeout):
        for layer in m.list_layers():
            if layer["name"] == name and layer["id"] not in known:
                return layer["id"]
        await asyncio.sleep(1)
    raise TimeoutError(f"No new layer named {name!r} appeared within {timeout} seconds")


def layer_ids():
    """Return the ids of the layers on the map now."""
    return {layer["id"] for layer in m.list_layers()}


if HAS_LAYER_IDS:
    known = layer_ids()
    m.run_whitebox_tool("slope", input=elevation_id, output="slope.tif")
    whitebox_slope_id = await wait_for_new_layer("Slope output", known)
    m.zoom_to_layer(whitebox_slope_id)

## 3.3 Run a processing tool on a vector layer

`run_algorithm` runs a built-in tool by its id. Its parameters match the tool's dialog in the **Processing** menu, with `layer` taking a layer id. This one draws a 300 km zone around each of the ten strongest earthquakes.

In [ ]:
if HAS_LAYER_IDS:
    known = layer_ids()
    m.run_algorithm("buffer", layer=strongest_id, distance=300, units="kilometers")
    buffer_id = await wait_for_new_layer("Buffer", known)
    m.set_style(buffer_id, fillColor="#22d3ee", fillOpacity=0.15, strokeColor="#22d3ee")

## 3.4 Chain tools with a model

A Model Builder graph runs several tools as one pipeline. The nodes are the steps, and the edges connect one step's output to the next step's input. This model buffers the ten strongest earthquakes, then merges the zones that overlap.

Instead of writing a graph by hand, you can draw it in **Processing → Model builder** and use **Copy Python script**.

In [ ]:
if HAS_LAYER_IDS:
    graph = {
        "nodes": [
            {"id": "source", "kind": "input", "x": 0, "y": 0, "layerId": strongest_id},
            {
                "id": "buffer",
                "kind": "tool",
                "x": 240,
                "y": 0,
                "provider": "vector",
                "toolId": "buffer",
                "parameters": {"distance": 500, "units": "kilometers"},
            },
            {
                "id": "dissolve",
                "kind": "tool",
                "x": 480,
                "y": 0,
                "provider": "vector",
                "toolId": "dissolve",
            },
            {"id": "result", "kind": "output", "x": 720, "y": 0, "name": "Merged 500 km zones"},
        ],
        "edges": [
            {"id": "e1", "from": "source", "fromPort": "out", "to": "buffer", "toPort": "layer"},
            {"id": "e2", "from": "buffer", "fromPort": "out", "to": "dissolve", "toPort": "layer"},
            {"id": "e3", "from": "dissolve", "fromPort": "out", "to": "result", "toPort": "in"},
        ],
    }
    known = layer_ids()
    m.run_model_builder(graph)
    zones_id = await wait_for_new_layer("Merged 500 km zones", known)
    m.set_style(zones_id, fillColor="#a78bfa", fillOpacity=0.2, strokeColor="#a78bfa")

## 3.5 Clean up

`remove_layer` deletes a layer by its id. On the web, remove layers from the Layers panel instead.

In [ ]:
if HAS_LAYER_IDS:
    # Everything added since Part 3 began, including the model's intermediate "Buffer".
    for layer_id in layer_ids() - part3_start | {summary_id, heatmap_id}:
        m.remove_layer(layer_id)
    print("Remaining layers:", [layer["name"] for layer in m.list_layers()])

## Where to go next

- The [Notebook panel docs](https://geolibre.org/notebook/) describe this client and how to connect an external editor such as VS Code to the desktop server.
- The [`geolibre` Python package](https://geolibre.org/python/) embeds the whole app in any Jupyter notebook and authors `.geolibre.json` project files.
- The [Python Console](https://geolibre.org/user-guide/python-console/) has a larger scripting API, including results you can read back on the web.